<a href="https://colab.research.google.com/github/np03cs4a240372-tech/AI/blob/main/Worksheet_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Worksheet-8


## Question 1: Custom Decision Tree vs Scikit-Learn (Iris Dataset)
This section compares a hand-built decision tree with a library-based implementation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
class CustomDecisionTree:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        self.tree = self._build_tree(X, y)

    def _entropy(self, y):
        probs = np.bincount(y) / len(y)
        return -np.sum(probs * np.log2(probs + 1e-9))

    def _information_gain(self, parent, left, right):
        return self._entropy(parent) - (
            len(left)/len(parent)*self._entropy(left) +
            len(right)/len(parent)*self._entropy(right)
        )

    def _build_tree(self, X, y, depth=0):
        if len(set(y)) == 1:
            return {'class': y[0]}
        if self.max_depth and depth >= self.max_depth:
            return {'class': np.bincount(y).argmax()}

        best_gain = -1
        best_split = None

        for f in range(X.shape[1]):
            for t in np.unique(X[:, f]):
                left = y[X[:, f] <= t]
                right = y[X[:, f] > t]
                gain = self._information_gain(y, left, right)
                if gain > best_gain:
                    best_gain = gain
                    best_split = (f, t)

        if best_split is None:
            return {'class': np.bincount(y).argmax()}

        f, t = best_split
        idx = X[:, f] <= t

        return {
            'feature': f,
            'threshold': t,
            'left': self._build_tree(X[idx], y[idx], depth+1),
            'right': self._build_tree(X[~idx], y[~idx], depth+1)
        }

    def predict(self, X):
        return [self._predict(x, self.tree) for x in X]

    def _predict(self, x, node):
        if 'class' in node:
            return node['class']
        if x[node['feature']] <= node['threshold']:
            return self._predict(x, node['left'])
        return self._predict(x, node['right'])

In [18]:
data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 120
Testing samples: 30


In [ ]:
custom = CustomDecisionTree(max_depth=3)
custom.fit(X_train, y_train)
acc_custom = accuracy_score(y_test, custom.predict(X_test))
acc_custom

1.0

In [ ]:
sk = DecisionTreeClassifier(max_depth=3, random_state=42)
sk.fit(X_train, y_train)
acc_sk = accuracy_score(y_test, sk.predict(X_test))
acc_sk

1.0

## Question 2: Ensemble Classification (Wine Dataset)
Comparison between Decision Tree and Random Forest using F1-score.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

f1_dt = f1_score(y_test, dt.predict(X_test), average='weighted')
f1_rf = f1_score(y_test, rf.predict(X_test), average='weighted')

f1_dt, f1_rf

(0.9439974457215836, 1.0)

## Question 3: Hyperparameter Tuning using GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'n_estimators': [50, 100],
    'max_depth': [None, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), params, cv=5)
grid.fit(X_train, y_train)
grid.best_params_

{'max_depth': None, 'n_estimators': 100}

## Question 4: Regression using Decision Tree & Random Forest (Diabetes Dataset)

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt_reg = DecisionTreeRegressor(random_state=42)
rf_reg = RandomForestRegressor(random_state=42)

dt_reg.fit(X_train, y_train)
rf_reg.fit(X_train, y_train)

mse_dt = mean_squared_error(y_test, dt_reg.predict(X_test))
mse_rf = mean_squared_error(y_test, rf_reg.predict(X_test))

mse_dt, mse_rf

(4976.797752808989, 2952.0105887640448)